<a href="https://colab.research.google.com/github/SubhadeepBhadra/subhflyrank-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SubhadeepBhadra/subhflyrank-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: CTR / Engagement Opportunity Scoring (Lane 4).**

I'm picking this lane because the starter dataset already shows a large, easy-to-verify gap in click-through rate across position tiers (see Section 3), which means there is a real, measurable pattern to score against — not something I have to invent. It also produces a concrete, useful output (a ranked list of pages that are under-capturing clicks for their position) rather than an abstract report, which keeps the whole project tied to an action someone could actually take.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

For an SEO content editor deciding which published pages to review first, we will build a ranked opportunity list from 90-day search performance data (impressions, clicks, CTR, position), scoring how far a page's CTR falls below the average CTR of other pages in the same position tier. The editor acts by opening the top-ranked pages and rewriting the title/meta description or improving the search snippet.

A wrong call costs two ways: a false positive sends an editor to rewrite a page whose low CTR is actually normal for its content type or has too little traffic volume to be meaningful — wasted editor hours. A false negative means a page that is genuinely under-capturing clicks for its position (and losing traffic it could otherwise get) goes unreviewed. A plain fixed CTR threshold isn't enough because expected CTR depends heavily on position — comparing a position-9 page and a position-1 page on raw CTR alone is misleading.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/SubhadeepBhadra/subhflyrank-internship.git
%cd subhflyrank-internship//work/notebooks
import pandas as pd

# Load the starter dataset (path is relative to work/notebooks/)
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Shape:", df.shape)

# Number 1: mean CTR by position tier -- the core premise of this lane
ctr_by_tier = df.groupby("position_tier")["ctr"].mean().round(3).sort_values(ascending=False)
print("\nMean CTR (%) by position tier:")
print(ctr_by_tier)

# Number 2: the gap between the best and worst tier
gap_ratio = round(ctr_by_tier.max() / ctr_by_tier.min(), 1)
print(f"\nCTR gap: top tier is about {gap_ratio}x the CTR of the lowest tier")

# Number 3: how many pages have enough volume to score reliably (avoid low-volume noise)
enough_volume = (df["impressions_90d"] >= 100).sum()
print(f"\nPages with >= 100 impressions in 90 days (usable volume): {enough_volume} of {len(df)} "
      f"({enough_volume/len(df):.1%})")

# Note: avg_position == 0 means "no data", not rank zero -- flag it, don't treat as a real position
no_position_data = (df["avg_position"] == 0).sum()
print(f"Rows with avg_position == 0 (no data, excluded from scoring): {no_position_data}")

fatal: destination path 'subhflyrank-internship' already exists and is not an empty directory.
/content/subhflyrank-internship/work/notebooks
Shape: (30000, 44)

Mean CTR (%) by position tier:
position_tier
top_3       1.484
page_1      0.652
striking    0.323
page_3_5    0.222
deep        0.150
Name: ctr, dtype: float64

CTR gap: top tier is about 9.9x the CTR of the lowest tier

Pages with >= 100 impressions in 90 days (usable volume): 22006 of 30000 (73.4%)
Rows with avg_position == 0 (no data, excluded from scoring): 1205


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick look numbers for section 3
import pandas as pd

try:
    df
except NameError:
    df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

top3_ctr = df.loc[df['position_tier'] == 'top_3', 'ctr'].mean()
deep_ctr = df.loc[df['position_tier'] == 'deep', 'ctr'].mean()
top3_count = (df['position_tier'] == 'top_3').sum()
deep_count = (df['position_tier'] == 'deep').sum()
rows_with_100 = (df['impressions_90d'] >= 100).sum()
avg_position_zero = (df['avg_position'] == 0).sum()
print(f'Rows: {len(df)}')
print(f'Mean CTR top_3: {top3_ctr:.3%} (n={top3_count})')
print(f'Mean CTR deep: {deep_ctr:.3%} (n={deep_count})')
print(f'Top_3 vs deep CTR ratio: {top3_ctr / deep_ctr:.1f}x')
print(f'Pages with >=100 impressions in 90 days: {rows_with_100} ({rows_with_100 / len(df):.1%})')
print(f'Rows with avg_position == 0: {avg_position_zero} ({avg_position_zero / len(df):.1%})')


Rows: 30000
Mean CTR top_3: 148.361% (n=2321)
Mean CTR deep: 15.021% (n=1319)
Top_3 vs deep CTR ratio: 9.9x
Pages with >=100 impressions in 90 days: 22006 (73.4%)
Rows with avg_position == 0: 1205 (4.0%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What this work CAN say: which pages show an **observed** CTR gap relative to other pages at the same position tier, ranked as **decision-support** for an editor's review queue, with the direction (under- or over-performing) clearly labeled and volume-filtered so noise from very low-impression pages doesn't dominate the ranking.

What this work will NEVER say: that a low CTR was *caused* by a specific title, meta description, or content issue (that requires an experiment, not an observational score); that any result reveals or predicts how Google's ranking algorithm works; or that acting on the ranked list guarantees a click or traffic increase. These are association-based rankings, not causal proof.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Sanity check: confirm no client-identifying or raw text fields are used anywhere above
unsafe_cols = [c for c in df.columns if c.lower() in ("client_name", "url", "domain", "query_text")]
print("Unsafe columns present in this dataframe:", unsafe_cols if unsafe_cols else "none")
print("content_id / client_id are pseudonyms used only for grouping, never as model features.")

Unsafe columns present in this dataframe: none
content_id / client_id are pseudonyms used only for grouping, never as model features.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.